# ⚡ LightningChess — Training & Benchmarks

This notebook walks through:
1. Visualising PGN dataset statistics
2. Training the MLP and CNN models inline
3. Exporting to TorchScript for LibTorch
4. Benchmarking GPU vs CPU inference speed
5. Generating ELO-progression and nodes/sec graphs

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'train'))

import torch
import matplotlib.pyplot as plt
import numpy as np

print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Import training utilities

In [ ]:
from train import MLP, CNN, board_to_flat, board_to_tensor, parse_pgn
import chess

mlp = MLP()
cnn = CNN()
print(f'MLP parameters: {mlp.param_count():,}')
print(f'CNN parameters: {cnn.param_count():,}')

## 2. Quick sanity check — encode starting position

In [ ]:
board = chess.Board()
flat  = board_to_flat(board)
cnn_t = board_to_tensor(board)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
plane_names = [
    'W-Pawn','W-Knight','W-Bishop','W-Rook','W-Queen','W-King',
    'B-Pawn','B-Knight','B-Bishop','B-Rook','B-Queen','B-King',
]
for i, ax in enumerate(axes.flat):
    ax.imshow(cnn_t[i].numpy(), cmap='Blues', vmin=0, vmax=1)
    ax.set_title(plane_names[i], fontsize=9)
    ax.axis('off')
plt.suptitle('Board encoding — 12 planes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('board_encoding.png', dpi=150)
plt.show()

## 3. Train (small demo — replace path with your PGN)

In [ ]:
PGN_PATH = '../data/games.pgn'   # ← point to your PGN file
EPOCHS   = 20                    # increase for real training
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

import os
if not os.path.exists(PGN_PATH):
    print(f'PGN not found at {PGN_PATH}.')
    print('Download from https://database.lichess.org/ or generate with Stockfish self-play.')
else:
    from train import train_model, export_model
    from torch.utils.data import DataLoader, TensorDataset

    flat_boards, cnn_boards, scores = parse_pgn(PGN_PATH, max_positions=50_000)
    y = torch.tensor(scores, dtype=torch.float32)

    mlp_loader = DataLoader(TensorDataset(torch.stack(flat_boards), y),
                             batch_size=512, shuffle=True)
    cnn_loader = DataLoader(TensorDataset(torch.stack(cnn_boards), y),
                             batch_size=512, shuffle=True)

    train_model(mlp, mlp_loader, EPOCHS, device=DEVICE, name='MLP')
    train_model(cnn, cnn_loader, EPOCHS, device=DEVICE, name='CNN')

## 4. Export to TorchScript

In [ ]:
os.makedirs('../models', exist_ok=True)

example_flat = torch.zeros(1, 768)
example_cnn  = torch.zeros(1, 12, 8, 8)

export_model(mlp, example_flat, '../models/mlp.pt')
export_model(cnn, example_cnn,  '../models/cnn.pt')

# Verify loading
loaded_mlp = torch.jit.load('../models/mlp.pt')
loaded_cnn = torch.jit.load('../models/cnn.pt')
print('MLP output:', loaded_mlp(example_flat).item())
print('CNN output:', loaded_cnn(example_cnn).item())
print('Export OK ✓')

## 5. GPU vs CPU inference benchmark

In [ ]:
import time

BATCH_SIZE = 256
N_ITER     = 200

def benchmark_model(model, example_input, device, n_iter=N_ITER):
    model = model.to(device).eval()
    x = example_input.to(device).repeat(BATCH_SIZE, *([1]*(example_input.dim()-1)))
    # Warmup
    with torch.no_grad():
        for _ in range(10):
            model(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_iter):
            model(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    pos_per_sec = BATCH_SIZE * n_iter / elapsed
    return pos_per_sec

cpu = torch.device('cpu')
gpu = torch.device('cuda') if torch.cuda.is_available() else None

results = {}
for model, name, example in [
    (mlp, 'MLP', example_flat),
    (cnn, 'CNN', example_cnn),
]:
    cpu_pps = benchmark_model(model, example, cpu)
    results[f'{name}-CPU'] = cpu_pps
    print(f'{name} CPU: {cpu_pps:,.0f} pos/sec')

    if gpu:
        gpu_pps = benchmark_model(model, example, gpu)
        results[f'{name}-GPU'] = gpu_pps
        print(f'{name} GPU: {gpu_pps:,.0f} pos/sec  ({gpu_pps/cpu_pps:.1f}×)')

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
labels = list(results.keys())
values = [v / 1e6 for v in results.values()]
colors = ['#4477ff' if 'CPU' in l else '#ff6644' for l in labels]
bars = ax.bar(labels, values, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
ax.set_ylabel('Positions / second (millions)', fontsize=11)
ax.set_title('⚡ LightningChess — GPU vs CPU Throughput', fontsize=13, fontweight='bold')
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{v:.2f}M', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim(0, max(values) * 1.25)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('benchmark_throughput.png', dpi=150)
plt.show()

## 6. Simulated ELO progression chart

In [ ]:
# Simulated ELO curve — replace with real data from self-play
epochs    = np.arange(0, 101, 5)
elo_hce   = np.full_like(epochs, 1800, dtype=float)
elo_mlp   = 1800 + 600 * (1 - np.exp(-epochs / 40))
elo_cnn   = 1800 + 900 * (1 - np.exp(-epochs / 35)) + np.random.normal(0, 15, len(epochs))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, elo_hce, '--', color='#aaaaaa', lw=2, label='HCE baseline')
ax.plot(epochs, elo_mlp, '-',  color='#4477ff', lw=2.5, label='MLP eval')
ax.plot(epochs, elo_cnn, '-',  color='#ff6644', lw=2.5, label='CNN eval')
ax.fill_between(epochs, elo_mlp, elo_hce, alpha=0.12, color='#4477ff')
ax.fill_between(epochs, elo_cnn, elo_mlp, alpha=0.12, color='#ff6644')
ax.set_xlabel('Training epoch', fontsize=11)
ax.set_ylabel('Estimated ELO', fontsize=11)
ax.set_title('⚡ LightningChess — ELO Progression', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('elo_progression.png', dpi=150)
plt.show()

## 7. Next steps

- Download a large PGN dataset from https://database.lichess.org/
- Run `python train/train.py --pgn data/games.pgn --epochs 100`
- Copy `models/mlp.pt` and `models/cnn.pt` next to the engine binary
- Enable CNN mode: `setoption name EvalMode value CNN`
- Watch the nodes/sec counter in the live dashboard!